# Feature Selection Pipeline

This notebook audits fold-safe feature selection before training the SRM Global Linear model.

**Research question**

Does selecting imaging features inside each training fold improve paired progression sensitivity compared with using all imaging features?

**Methods compared**

| Method | Full name | What it does | Leakage control |
|---|---|---|---|
| `none` | No feature selection | Uses all imaging features. | No selection is fitted. |
| `mi` | Mutual Information | Ranks features by visit-label information and keeps the top `k`. | Ranking is recomputed inside each training fold. |
| `mml` | Minimum Message Length | Greedy forward selection keeps features that reduce linear-regression message length. | Selection is recomputed inside each training fold. |

**Important concepts**

- **LOO:** Leave-One-Out evaluation. In this project, the held-out unit is the participant group, so all intervals from the same subject are held out together.
- **OOF:** Out-Of-Fold predictions. These are predictions for samples not used to train that fold's model.
- **Primary tuning metric:** mean annual paired Cohen's `d_z`, computed separately for held-out V1->V2 and V2->V3 interval scores, then averaged. The interval gap `abs(dz_V1_V2 - dz_V2_V3)` is a diagnostic and tie-breaker.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv, lda_loocv, tune_and_run_regression_loocv
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.metrics import bootstrap_ci_d, clinical_change_effect_sizes, paired_deltas_from_long, probability_positive_change, reference_effect_sizes
from src.eval.model_selection import select_hierarchical_candidate
from src.eval.stability import selected_feature_jaccard
from src.features.selection import feature_stability_report, select_one_se_candidate
from src.models.srm_global import srm_global_loocv, srm_global_repeated_group_cv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_k = 8
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 1000
RANDOM_SEED = DEFAULT_CONFIG.random_state
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows, {len(imaging_cols)} imaging features")
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)


Loaded trackfa_pairs_drop3poms.csv: 414 visit rows, 146 imaging features


## 1. Compare Selection Methods

Each run trains the same SRM Global Linear model. The only intended difference is the feature-selection method applied inside the outer training fold.

**How to read the table:** higher absolute `d_z` indicates stronger standardised progression change, but stability and leakage safety matter as much as the point estimate.


In [2]:
results = []
selected = {}
interval_results = {}
for method in ["mi", "mml", "none"]:
    res = srm_global_loocv(
        long_df,
        imaging_cols,
        subject_col=subject_col,
        visit_col="visit",
        selection_method=method,
        k=selection_k,
        cv_n_splits=CV_N_SPLITS,
        random_seed=RANDOM_SEED,
        split_group_col=split_group_col,
    )
    selected[method] = res["selected_features_by_fold"]
    annual_intervals = adjacent_pair_interval_effect_summary(
        res["oof_df"],
        pair_col=subject_col,
        visit_col="visit",
        score_col="score",
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    interval_results[method] = annual_intervals
    annual_diag = annual_tuning_diagnostics(annual_intervals)
    pooled_deltas = paired_deltas_from_long(res["oof_df"].rename(columns={"score": "value"}), subject_col, "visit", "value")
    results.append({
        "selection_method": method,
        "pooled_pair_d_z_reference": res["d_score"],
        "ci_low": res["d_ci_low"],
        "ci_high": res["d_ci_high"],
        **annual_diag,
        "pooled_p_progression_reference": probability_positive_change(pooled_deltas),
        "n_subject_pairs": res["n_subjects"],
        "median_n_features": np.median([len(x) for x in selected[method]]) if selected[method] else np.nan,
    })
selection_results = pd.DataFrame(results).sort_values(
    ["mean_validation_annual_dz", "annual_interval_gap"],
    ascending=[False, True],
)
print("Annual-interval tuning diagnostics by selector")
display(selection_results)
print("Intervals remain separately observable")
for method, table in interval_results.items():
    print(method)
    display(table)


Annual-interval tuning diagnostics by selector


,selection_method,pooled_pair_d_z_reference,ci_low,ci_high,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,pooled_p_progression_reference,n_subject_pairs,median_n_features
2,none,0.499799,0.362467,0.650368,0.592037,0.417083,0.504560,0.174954,0.715067,0.714976,207,146.0
1,mml,0.372425,0.243720,0.514890,0.433076,0.315533,0.374305,0.117543,0.669613,0.671498,207,97.0
0,mi,0.153240,0.015679,0.287735,0.257121,0.053542,0.155332,0.203579,0.587542,0.589372,207,8.0


Intervals remain separately observable
mi


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,0.080420,0.312770,0.257121,0.058608,0.477437,0.629630
1,V2->V3,99,0.018831,0.351706,0.053542,-0.141673,0.263415,0.545455


mml


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,0.879511,2.030845,0.433076,0.261423,0.629356,0.712963
1,V2->V3,99,0.739467,2.343549,0.315533,0.114880,0.514442,0.626263


none


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,4.265871,7.205419,0.592037,0.403467,0.797171,0.712963
1,V2->V3,99,3.565920,8.549666,0.417083,0.216794,0.659898,0.717172


## 2. Inspect Feature-Selection Stability

Feature selection can look strong by chance when the selected feature set changes wildly across folds. Retention rate shows how often a feature is kept, while mean Jaccard summarises overlap between fold-specific selected sets.


In [3]:
mi_stability = feature_stability_report(selected.get("mi", []), imaging_cols)
mml_stability = feature_stability_report(selected.get("mml", []), imaging_cols)
print("MI stability")
display(mi_stability.head(25))
print("MML stability")
display(mml_stability.head(25))

MI stability


,feature,retention_rate,mean_jaccard
0,FA_gCC,0.6,0.123297
1,MD_Cing_h,0.6,0.123297
2,MD_mLEM,0.6,0.123297
3,sCSA_C12_UMN,0.6,0.123297
4,CCtotal,0.4,0.123297
5,FA_mLEM,0.4,0.123297
6,MD_PCT,0.4,0.123297
7,sAD_c3c5,0.4,0.123297
8,sMD_c3c5,0.4,0.123297
9,AD_Cing_h,0.2,0.123297


MML stability


,feature,retention_rate,mean_jaccard
0,AD_ACR,1.0,0.917797
1,AD_ALIC,1.0,0.917797
2,AD_CP,1.0,0.917797
3,AD_CST,1.0,0.917797
4,AD_Cing,1.0,0.917797
5,AD_Cing_h,1.0,0.917797
6,AD_EC,1.0,0.917797
7,AD_Fx,1.0,0.917797
8,AD_Fx_ST,1.0,0.917797
9,AD_ICP,1.0,0.917797


## 2a. One-SE Selection Rule


In [4]:
# Apply the reusable annual-consistency one-SE rule to choose a simple, stable selector from notebook-visible results.
stability_lookup = {}
for method, table in {"mi": mi_stability, "mml": mml_stability}.items():
    stability_lookup[method] = float(table["mean_jaccard"].dropna().iloc[0]) if len(table.dropna(subset=["mean_jaccard"])) else np.nan
stability_lookup["none"] = selected_feature_jaccard(selected["none"])["mean_jaccard"] if "none" in selected else 1.0

one_se_table = selection_results.rename(columns={"median_n_features": "feature_count"}).copy()
one_se_table["se_validation_dz"] = (one_se_table["ci_high"] - one_se_table["ci_low"]) / (2 * 1.96)
one_se_table["jaccard_stability"] = one_se_table["selection_method"].map(stability_lookup)
one_se_table["sign_stability"] = np.nan
one_se_table["score_ranking_stability"] = np.nan
one_se_choice = select_hierarchical_candidate(one_se_table)
print("Annual-consistency hierarchical one-SE selected configuration")
print("Primary metric: mean annual d_z = mean(dz_V1_V2, dz_V2_V3). Tie-breakers: smaller interval gap, higher P(delta>0), fewer features, Jaccard/sign/ranking stability.")
display(pd.DataFrame([one_se_choice]))
display(one_se_table.sort_values(["mean_validation_annual_dz", "annual_interval_gap"], ascending=[False, True]))


Annual-consistency hierarchical one-SE selected configuration
Primary metric: mean annual d_z = mean(dz_V1_V2, dz_V2_V3). Tie-breakers: smaller interval gap, higher P(delta>0), fewer features, Jaccard/sign/ranking stability.


,selection_method,pooled_pair_d_z_reference,ci_low,ci_high,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,pooled_p_progression_reference,n_subject_pairs,feature_count,se_validation_dz,jaccard_stability,sign_stability,score_ranking_stability,coefficient_sign_stability,directional_consistency
2,none,0.499799,0.362467,0.650368,0.592037,0.417083,0.50456,0.174954,0.715067,0.714976,207,146.0,0.073444,1.0,-inf,-inf,-inf,-inf


,selection_method,pooled_pair_d_z_reference,ci_low,ci_high,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,pooled_p_progression_reference,n_subject_pairs,feature_count,se_validation_dz,jaccard_stability,sign_stability,score_ranking_stability
2,none,0.499799,0.362467,0.650368,0.592037,0.417083,0.504560,0.174954,0.715067,0.714976,207,146.0,0.073444,1.000000,NaN,NaN
1,mml,0.372425,0.243720,0.514890,0.433076,0.315533,0.374305,0.117543,0.669613,0.671498,207,97.0,0.069176,0.917797,NaN,NaN
0,mi,0.153240,0.015679,0.287735,0.257121,0.053542,0.155332,0.203579,0.587542,0.589372,207,8.0,0.069402,0.123297,NaN,NaN


## 3. Compare Against Clinical Benchmarks

The final table places the model's paired `d_z` beside FARS, SARA, and the strongest single imaging feature. Clinical benchmarks are computed from adjacent clinical changes only.


In [5]:
best = selection_results.iloc[0]
display(benchmark_table(f"SRM Global Linear ({best['selection_method']})", best["pooled_pair_d_z_reference"], best["ci_low"], best["ci_high"]))


,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types
0,SRM Global Linear (none),model,0.499799,0.362467,0.650368,NaN,NaN
1,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3"
2,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3"
3,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN
